In [ ]:
import numpy as np
import math
import time
import matplotlib.pyplot as plt

def build_circulant(n, c1, c2, c3):
    """
    Builds the structured circulant matrix C for verification.
    Assumes a tridiagonal layout where c2 is the main diagonal,
    c3 is the super-diagonal, and c1 is the sub-diagonal.
    """
    if n == 1:
        return np.array([[c1 + c2 + c3]])
    if n == 2:
        return np.array([
            [c2, c1 + c3],
            [c1 + c3, c2]
        ])

    C = np.zeros((n, n), dtype=int)
    for i in range(n):
        C[i, i] = c2
        C[i, (i + 1) % n] = c3
        C[i, (i - 1) % n] = c1
    return C

def ryser_permanent(A):
    """
    Computes the permanent of a matrix using Ryser's algorithm.
    Time Complexity: O(n^2 * 2^n).
    """
    n = len(A)
    if n == 0:
        return 0

    total = 0
    # Iterate through all non-empty subsets of columns
    for S in range(1, 1 << n):
        # Determine the sign based on the size of the subset
        sign = -1 if (n - bin(S).count('1')) % 2 != 0 else 1

        prod = 1
        for i in range(n):
            # Sum the elements in the i-th row for the selected columns
            row_sum = sum(A[i][j] for j in range(n) if (S & (1 << j)))
            prod *= row_sum
            if prod == 0:
                break
        total += sign * prod

    return total

def user_formula(n, c1, c2, c3):
    """
    Computes the permanent using the custom O(n) closed-form formula.
    """
    if n == 1:
        return c1 + c2 + c3

    term1 = c1**n + c2**n + c3**n
    sigma = 0

    # Python's integer division // naturally handles both n/2 (even) and (n-1)/2 (odd)
    limit = n // 2

    for r in range(1, limit + 1):
        if n - r > 0:
            coef = math.comb(n - r, r) * (n / (n - r))
            # Rounding to handle floating point precision artifacts in division
            coef = int(round(coef))
            sigma += coef * (c1**r) * (c2**(n - 2*r)) * (c3**r)

    return term1 + sigma

def run_comparison():
    # Parameters for the circulant matrix
    c1, c2, c3 = 2, 3, 4
    max_n = 50

    # Cap Ryser at n=16 to prevent the script from hanging infinitely
    ryser_cap = 16

    ns = list(range(1, max_n + 1))
    ryser_times = []
    user_times = []

    print(f"--- Verification Phase (n=1 to {ryser_cap}) ---")
    for n in range(1, ryser_cap + 1):
        C = build_circulant(n, c1, c2, c3)

        # Benchmark Ryser
        start = time.perf_counter()
        ryser_ans = ryser_permanent(C)
        r_time = time.perf_counter() - start
        ryser_times.append(r_time)

        # Benchmark User Formula
        start = time.perf_counter()
        user_ans = user_formula(n, c1, c2, c3)
        u_time = time.perf_counter() - start
        user_times.append(u_time)

        # Verify correctness
        is_match = (ryser_ans == user_ans)
        match_str = "MATCH" if is_match else "MISMATCH"
        print(f"n={n:2} | Ryser: {ryser_ans:<15} | Formula: {user_ans:<15} | Status: {match_str}")

        if not is_match:
            print("Mathematical divergence detected. Stopping execution.")
            return

    print(f"\n--- Extrapolation Phase (n={ryser_cap + 1} to {max_n}) ---")
    print("Computing remaining values using the closed-form formula...")

    for n in range(ryser_cap + 1, max_n + 1):
        # We append 'None' for Ryser to indicate it timed out / was skipped
        ryser_times.append(None)

        start = time.perf_counter()
        _ = user_formula(n, c1, c2, c3)
        u_time = time.perf_counter() - start
        user_times.append(u_time)

    print("Computations complete. Generating performance graph...")

    # Plotting the results
    plt.figure(figsize=(10, 6))

    # Plot Ryser (only up to the cap)
    valid_ryser_times = [t for t in ryser_times if t is not None]
    plt.plot(ns[:len(valid_ryser_times)], valid_ryser_times,
             marker='o', color='red', label="Ryser's Formula $O(n 2^n)$", linewidth=2)

    # Plot User Formula (all the way to max_n)
    plt.plot(ns, user_times,
             marker='s', color='green', label="Your Closed-Form Formula $O(n)$", linewidth=2)

    plt.yscale('log')
    plt.xlabel('Matrix Size (n x n)')
    plt.ylabel('Execution Time (seconds) [Log Scale]')
    plt.title('Performance Comparison: Ryser\'s vs. Custom Formula')
    plt.grid(True, which="both", ls="--", alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    run_comparison()